# CSV metadata customization walkthrough
This notebook provides sample code walkthrough for 'CSV metadata customization' feature, a feautre from Amazon Bedrock Knowledge bases which enhances .csv file processing feature that separates content and metadata. .

For more details on this feature, please read this [blog](https://aws.amazon.com/blogs/machine-learning/knowledge-bases-for-amazon-bedrock-now-supports-advanced-parsing-chunking-and-query-reformulation-giving-greater-control-of-accuracy-in-rag-based-applications/#:~:text=Machine%20Learning%20Blog-,Knowledge%20Bases%20for%20Amazon%20Bedrock%20now%20supports%20advanced%20parsing%2C%20chunking,accuracy%20in%20RAG%20based%20applications).

## 1. Import the needed libraries
First step is to install the pre-requisites packages.

In [ ]:
%pip install --upgrade pip
%pip install -r ../requirements.txt

In [ ]:
# restart kernel
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import botocore
botocore.__version__

This code is part of the setup and used to :
- Add the parent directory to the python system path
- Imports a custom module (BedrockStructuredKnowledgeBase) from `utils` necessary for later executions

In [ ]:
import sys
import logging
from pathlib import Path
import os
import time
import boto3
import pprint
import json

current_path = Path().resolve()
current_path = current_path.parent

if str(current_path) not in sys.path:
    sys.path.append(str(current_path))

# Print sys.path to verify
print(sys.path)

from utils.knowledge_base import BedrockKnowledgeBase

In [ ]:
#Clients
s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region =  session.region_name
account_id = sts_client.get_caller_identity()["Account"]
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime') 
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)
region, account_id

('us-west-2', '166151834296')

In [ ]:
import time

# Get the current timestamp
current_time = time.time()

# Format the timestamp as a string
timestamp_str = time.strftime("%Y%m%d%H%M%S", time.localtime(current_time))[-7:]
# Create the suffix using the timestamp
suffix = f"{timestamp_str}"
knowledge_base_name_standard = 'csv-metadata-kb'
knowledge_base_name_hierarchical = 'hierarchical-kb'
knowledge_base_description = "Knowledge Base csv metadata customization."
bucket_name = f'{knowledge_base_name_standard}-{suffix}'
foundation_model = "global.anthropic.claude-sonnet-4-6"

# Define data sources
data_source=[{"type": "S3", "bucket_name": bucket_name}]

## 2 - Create knowledge bases with fixed chunking strategy
Let's start by creating a [Amazon Bedrock Knowledge Bases](https://aws.amazon.com/bedrock/knowledge-bases/) to store video games data in csv format. Knowledge Bases allow you to integrate with different vector databases including [Amazon OpenSearch Serverless](https://aws.amazon.com/opensearch-service/features/serverless/), [Amazon Aurora](https://aws.amazon.com/rds/aurora/), [Pinecone](http://app.pinecone.io/bedrock-integration), [Redis Enterprise]() and [MongoDB Atlas](). For this example, we will integrate the knowledge base with Amazon OpenSearch Serverless. To do so, we will use the helper class `BedrockKnowledgeBase` which will create the knowledge base and all of its pre-requisites:
1. IAM roles and policies
2. S3 bucket
3. Amazon OpenSearch Serverless encryption, network and data access policies
4. Amazon OpenSearch Serverless collection
5. Amazon OpenSearch Serverless vector index
6. Knowledge base
7. Knowledge base data source

We will create a knowledge base using fixed chunking strategy. 

You can chhose different chunking strategies by changing the below parameter values: 
```
"chunkingStrategy": "FIXED_SIZE | NONE | HIERARCHICAL | SEMANTIC"
```

In [ ]:
knowledge_base_standard = BedrockKnowledgeBase(
    kb_name=f'{knowledge_base_name_standard}-{suffix}',
    kb_description=knowledge_base_description,
    data_sources=data_source, 
    chunking_strategy = "FIXED_SIZE", 
    suffix = suffix
)

Step 1 - Creating or retrieving S3 bucket(s) for Knowledge Base documents
['csv-metadata-kb-1232323']
buckets_to_check:  ['csv-metadata-kb-1232323']
Creating bucket csv-metadata-kb-1232323


Step 2 - Creating Knowledge Base Execution Role (AmazonBedrockExecutionRoleForKnowledgeBase_1232323) and Policies


Step 3a - Creating OSS encryption, network and data access policies


Step 3b - Creating OSS Collection (this step takes a couple of minutes to complete)


{ 'ResponseMetadata': { 'HTTPHeaders': { 'connection': 'keep-alive',
                                         'content-length': '350',
                                         'content-type': 'application/x-amz-json-1.0',
                                         'date': 'Wed, 01 Jul 2026 23:23:44 '
                                                 'GMT',
                                         'x-amzn-requestid': '05207cc5-6ab9-419b-8f47-943e609cff79'},
                        'HTTPStatusCode': 200,
                        'RequestId': '05207cc5-6ab9-419b-8f47-943e609cff79',
                        'RetryAttempts': 0},
  'createCollectionDetail': { 'arn': 'arn:aws:aoss:us-west-2:166151834296:collection/ao6qemcgr4wk4pw7eh1g',
                              'createdDate': 1782948224166,
                              'deletionProtection': 'DISABLED',
                              'id': 'ao6qemcgr4wk4pw7eh1g',
                              'kmsKeyArn': 'auto',
                              

...

[2026-07-01 23:25:15,313] p157552 {base.py:258} INFO - PUT https://ao6qemcgr4wk4pw7eh1g.us-west-2.aoss.amazonaws.com:443/bedrock-sample-rag-index-1232323 [status:200 request:0.382s]



Creating index:
{ 'acknowledged': True,
  'index': 'bedrock-sample-rag-index-1232323',
  'shards_acknowledged': True}


Step 4 - Will create Lambda Function if chunking strategy selected as CUSTOM
Not creating lambda function as chunking strategy is FIXED_SIZE
Step 5 - Creating Knowledge Base


{ 'createdAt': datetime.datetime(2026, 7, 1, 23, 26, 15, 443325, tzinfo=tzlocal()),
  'description': 'Knowledge Base csv metadata customization.',
  'knowledgeBaseArn': 'arn:aws:bedrock:us-west-2:166151834296:knowledge-base/XFXDQPPIGX',
  'knowledgeBaseConfiguration': { 'type': 'VECTOR',
                                  'vectorKnowledgeBaseConfiguration': { 'embeddingModelArn': 'arn:aws:bedrock:us-west-2::foundation-model/amazon.titan-embed-text-v2:0'}},
  'knowledgeBaseId': 'XFXDQPPIGX',
  'name': 'csv-metadata-kb-1232323',
  'roleArn': 'arn:aws:iam::166151834296:role/AmazonBedrockExecutionRoleForKnowledgeBase_1232323',
  'status': 'CREATING',
  'storageConfiguration': { 'opensearchServerlessConfiguration': { 'collectionArn': 'arn:aws:aoss:us-west-2:166151834296:collection/ao6qemcgr4wk4pw7eh1g',
                                                                   'fieldMapping': { 'metadataField': 'text-metadata',
                                                                        

{ 'createdAt': datetime.datetime(2026, 7, 1, 23, 26, 15, 443325, tzinfo=tzlocal()),
  'description': 'Knowledge Base csv metadata customization.',
  'knowledgeBaseArn': 'arn:aws:bedrock:us-west-2:166151834296:knowledge-base/XFXDQPPIGX',
  'knowledgeBaseConfiguration': { 'type': 'VECTOR',
                                  'vectorKnowledgeBaseConfiguration': { 'embeddingModelArn': 'arn:aws:bedrock:us-west-2::foundation-model/amazon.titan-embed-text-v2:0'}},
  'knowledgeBaseId': 'XFXDQPPIGX',
  'name': 'csv-metadata-kb-1232323',
  'roleArn': 'arn:aws:iam::166151834296:role/AmazonBedrockExecutionRoleForKnowledgeBase_1232323',
  'status': 'ACTIVE',
  'storageConfiguration': { 'opensearchServerlessConfiguration': { 'collectionArn': 'arn:aws:aoss:us-west-2:166151834296:collection/ao6qemcgr4wk4pw7eh1g',
                                                                   'fieldMapping': { 'metadataField': 'text-metadata',
                                                                          

### 2.1 Download csv dataset and upload it to Amazon S3
Now that we have created the knowledge base, let's populate it with the `video_games.csv` dataset to KB. This data is being downloaded from [here](https://github.com/ali-ce/datasets/blob/master/Most-Expensive-Things/Videogames.csv). It contains the sales data of video games originally collected by Alice Corona is licensed under a [Creative Commons Attribution-ShareAlike 4.0 International License](https://github.com/ali-ce/datasets/blob/master/README.md#:~:text=Creative%20Commons%20Attribution%2DShareAlike%204.0%20International%20License.).


The Knowledge Base data source expects the data to be available on the S3 bucket connected to it and changes on the data can be syncronized to the knowledge base using the `StartIngestionJob` API call. In this example we will use the [boto3 abstraction](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent/client/start_ingestion_job.html) of the API, via our helper classes. 

In [ ]:
!mkdir -p ./csv_data

In [ ]:
!wget https://raw.githubusercontent.com/ali-ce/datasets/master/Most-Expensive-Things/Videogames.csv --no-check-certificate -O ./csv_data/video_games.csv

--2026-07-01 23:26:28--  https://raw.githubusercontent.com/ali-ce/datasets/master/Most-Expensive-Things/Videogames.csv


Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 23931 (23K) [text/plain]
Saving to: ‘./csv_data/video_games.csv’

./csv_data/video_ga 100%[===================>]  23.37K  --.-KB/s    in 0.002s  

2026-07-01 23:26:28 (11.8 MB/s) - ‘./csv_data/video_games.csv’ saved [23931/23931]



Let's upload the video games data available on the `csv_data` folder to s3.

In [ ]:
def upload_directory(path, bucket_name):
        for root,dirs,files in os.walk(path):
            for file in files:
                file_to_upload = os.path.join(root,file)
                print(f"uploading file {file_to_upload} to {bucket_name}")
                s3_client.upload_file(file_to_upload,bucket_name,file)

upload_directory("csv_data", bucket_name)

uploading file csv_data/video_games.csv to csv-metadata-kb-1232323


Now we start the ingestion job.

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_standard.start_ingestion_job()

job 1 started successfully



{ 'dataSourceId': 'EKF09JEXZN',
  'ingestionJobId': 'AEJTSF385J',
  'knowledgeBaseId': 'XFXDQPPIGX',
  'startedAt': datetime.datetime(2026, 7, 1, 23, 27, 14, 738087, tzinfo=tzlocal()),
  'statistics': { 'numberOfDocumentsDeleted': 0,
                  'numberOfDocumentsFailed': 0,
                  'numberOfDocumentsScanned': 1,
                  'numberOfDocumentsSkipped': 0,
                  'numberOfMetadataDocumentsModified': 0,
                  'numberOfMetadataDocumentsScanned': 0,
                  'numberOfModifiedDocumentsIndexed': 0,
                  'numberOfNewDocumentsIndexed': 1},
  'status': 'COMPLETE',
  'updatedAt': datetime.datetime(2026, 7, 1, 23, 27, 18, 509531, tzinfo=tzlocal())}


Finally we save the Knowledge Base Id to test the solution at a later stage. 

In [ ]:
kb_id_standard = knowledge_base_standard.get_knowledge_base_id()

'XFXDQPPIGX'


### 2.2 Query the Knowledge Base with Retrieve and Generate API - WITHOUT metadata

Let's test the knowledge base using the [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html) API. With this API, Bedrock takes care of retrieving the necessary references from the knowledge base and generating the final answer using a foundation model from Bedrock.

'''
query = "List the video games published by Rockstar Games and released after 2010"
'''

In [ ]:
query = "Provide a list of all video games published by Rockstar Games and released after 2010"

In [ ]:
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_standard,
            "modelArn": "arn:aws:bedrock:{}:{}:inference-profile/{}".format(region, account_id, foundation_model),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

pprint.pp(response['output']['text'])

('The search results do not contain any video games published by Rockstar '
 'Games that were released after 2010. The only Rockstar Games title mentioned '
 'in the results is Grand Theft Auto IV, which was released in 2008.')


The expected result is: Grand Theft Auto V, L.A. Noire, Max Payne 3

However, it is likely that nothing is returned. This is because when you ingest a CSV file with fixed chunking, the file is split into arbitrary text blocks of equal size (e.g., 300 or 500 characters) with no awareness of the data's structure. This causes three critical problems:
* Loss of column context
* Records split across chunk boundaries
* Semanatic search retrieves the wrong chunks

Next we are going to implement metadata mapping where we attach structured fields as filterable metadata to each chunk rather than relying on the embedding search alone.

#### 2.3 Prepeare metadata for ingestion


In [ ]:
import csv
import json

In [ ]:
def generate_json_metadata(csv_file, content_field, metadata_fields, excluded_fields):
    # Open the CSV file and read its contents
    with open(csv_file, 'r') as file:
        reader = csv.DictReader(file)
        headers = reader.fieldnames

    # Create the JSON structure
    json_data = {
        "metadataAttributes": {},
        "documentStructureConfiguration": {
            "type": "RECORD_BASED_STRUCTURE_METADATA",
            "recordBasedStructureMetadata": {
                "contentFields": [
                    {
                        "fieldName": content_field
                    }
                ],
                "metadataFieldsSpecification": {
                    "fieldsToInclude": [],
                    "fieldsToExclude": []
                }
            }
        }
    }

    # Add metadata fields to include
    for field in metadata_fields:
        json_data["documentStructureConfiguration"]["recordBasedStructureMetadata"]["metadataFieldsSpecification"]["fieldsToInclude"].append(
            {
                "fieldName": field
            }
        )

    # Add fields to exclude (all fields not in content_field or metadata_fields)
    if not excluded_fields:
        excluded_fields = set(headers) - set([content_field] + metadata_fields)
    
    for field in excluded_fields:
        json_data["documentStructureConfiguration"]["recordBasedStructureMetadata"]["metadataFieldsSpecification"]["fieldsToExclude"].append(
            {
                "fieldName": field
            }
        )

    # Generate the output JSON file name
    output_file = f"{csv_file.split('.')[0]}.csv.metadata.json"

    # Write the JSON data to the output file
    with open(output_file, 'w') as file:
        json.dump(json_data, file, indent=4)

    print(f"JSON metadata file '{output_file}' has been generated.")

In [ ]:
csv_file = 'csv_data/video_games.csv'
content_field = 'Videogame'
metadata_fields = ['Year', 'Developer', 'Publisher']
excluded_fields =['Description']

generate_json_metadata(csv_file, content_field, metadata_fields, excluded_fields)

JSON metadata file 'csv_data/video_games.csv.metadata.json' has been generated.


In [ ]:
# upload metadata file to S3
upload_directory("csv_data", bucket_name)

# delete metadata file from local
os.remove('csv_data/video_games.csv.metadata.json')

uploading file csv_data/video_games.csv.metadata.json to csv-metadata-kb-1232323
uploading file csv_data/video_games.csv to csv-metadata-kb-1232323
uploading file csv_data/.ipynb_checkpoints/video_games-checkpoint.csv to csv-metadata-kb-1232323


Now start the ingestion job. Since, we are using the same documents as used for fixed chunking, we are skipping the step to upload documents to s3 bucket. 

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_standard.start_ingestion_job()

job 1 started successfully



{ 'dataSourceId': 'EKF09JEXZN',
  'ingestionJobId': 'X93PT8YWWT',
  'knowledgeBaseId': 'XFXDQPPIGX',
  'startedAt': datetime.datetime(2026, 7, 1, 23, 59, 13, 491811, tzinfo=tzlocal()),
  'statistics': { 'numberOfDocumentsDeleted': 0,
                  'numberOfDocumentsFailed': 0,
                  'numberOfDocumentsScanned': 2,
                  'numberOfDocumentsSkipped': 0,
                  'numberOfMetadataDocumentsModified': 0,
                  'numberOfMetadataDocumentsScanned': 1,
                  'numberOfModifiedDocumentsIndexed': 1,
                  'numberOfNewDocumentsIndexed': 1},
  'status': 'COMPLETE',
  'updatedAt': datetime.datetime(2026, 7, 1, 23, 59, 20, 488372, tzinfo=tzlocal())}


### 2.4 Query the Knowledge Base with Retrieve and Generate API - WITH metadata

Create the filter 

In [ ]:
one_group_filter= {
    "andAll": [
        {
            "equals": {
                "key": "Publisher",
                "value": "Rockstar Games"
            }
        },
        {
            "greaterThan": {
                "key": "Year",
                "value": 2010
            }
        }
    ]
}

Pass the filter to `retrievalConfiguration` of the [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html).

In [ ]:
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_standard,
            "modelArn": "arn:aws:bedrock:{}:{}:inference-profile/{}".format(region, account_id, foundation_model),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5,
                    "filter": one_group_filter
                } 
            }
        }
    }
)

print(response['output']['text'])

The following video games were published by Rockstar Games and released after 2010:

1. L.A. Noire
2. Max Payne 3
3. Grand Theft Auto V


As you can see, we now get the correct results as there is additional context (the metadata) used in the retrieval process, rather than just purely relying on the semantic search.

Let's now observe the retrieved chunks and citations returned by the model while generating the response. When we provide the relevant context to the foundation model alongwith the query, it will most likely generate the high quality response. 

In [ ]:
response_standard = response['citations'][0]['retrievedReferences']
print("# of citations or chunks used to generate the response: ", len(response_standard))
def citations_rag_print(response_ret):
#structure 'retrievalResults': list of contents. Each list has content, location, score, metadata
    for num,chunk in enumerate(response_ret,1):
        print(f'Chunk {num}: ',chunk['content']['text'],end='\n'*2)
        print(f'Chunk {num} Location: ',chunk['location'],end='\n'*2)
        print(f'Chunk {num} Metadata: ',chunk['metadata'],end='\n'*2)

citations_rag_print(response_standard)

# of citations or chunks used to generate the response:  3
Chunk 1:  L.A. Noire

Chunk 1 Location:  {'s3Location': {'uri': 's3://csv-metadata-kb-1232323/video_games.csv'}, 'type': 'S3'}

Chunk 1 Metadata:  {'x-amz-bedrock-kb-source-uri': 's3://csv-metadata-kb-1232323/video_games.csv', 'Year': '2011', 'x-amz-bedrock-kb-data-source-id': 'EKF09JEXZN', 'x-amz-bedrock-kb-source-file-modality': 'TEXT', 'Developer': 'Team Bondi', 'Publisher': 'Rockstar Games', 'x-amz-bedrock-kb-chunk-id': '1%3A0%3AsOMfIJ8BLmYptL42Z7TD'}

Chunk 2:  Max Payne 3

Chunk 2 Location:  {'s3Location': {'uri': 's3://csv-metadata-kb-1232323/video_games.csv'}, 'type': 'S3'}

Chunk 2 Metadata:  {'x-amz-bedrock-kb-source-uri': 's3://csv-metadata-kb-1232323/video_games.csv', 'Year': '2012', 'x-amz-bedrock-kb-data-source-id': 'EKF09JEXZN', 'x-amz-bedrock-kb-source-file-modality': 'TEXT', 'Developer': 'Rockstar Studios', 'Publisher': 'Rockstar Games', 'x-amz-bedrock-kb-chunk-id': '1%3A0%3AseMfIJ8BLmYptL42Z7TD'}

Chunk 3:  Gr

In [ ]:
%store kb_id_standard

Stored 'kb_id_standard' (str)


### Clean up
Please make sure to uncomment and run below cells to delete the resources created in this notebook. If you are planning to run `dynamic-metadata-filtering` notebook under `03-advanced-concepts` section, then make sure to come back here to delete the resources. 

In [ ]:
# # Empty and delete S3 Bucket

# objects = s3_client.list_objects(Bucket=bucket_name)  
# if 'Contents' in objects:
#     for obj in objects['Contents']:
#         s3_client.delete_object(Bucket=bucket_name, Key=obj['Key']) 
# s3_client.delete_bucket(Bucket=bucket_name)

In [ ]:
# print("===============================Knowledge base==============================")
knowledge_base_standard.delete_kb(delete_s3_bucket=True, delete_iam_roles_and_policies=True)

Deleted data source EKF09JEXZN


======== Knowledge base and all data sources deleted =========
Found bucket csv-metadata-kb-1232323


Deleted all objects in bucket csv-metadata-kb-1232323


Deleted bucket csv-metadata-kb-1232323
======== S3 bucket deletion process completed =========
Found role AmazonBedrockExecutionRoleForKnowledgeBase_1232323


======Attached policies with role AmazonBedrockExecutionRoleForKnowledgeBase_1232323========
 [{'PolicyName': 'AmazonBedrockFoundationModelPolicyForKnowledgeBase_1232323', 'PolicyArn': 'arn:aws:iam::166151834296:policy/AmazonBedrockFoundationModelPolicyForKnowledgeBase_1232323'}, {'PolicyName': 'AmazonBedrockS3PolicyForKnowledgeBase_1232323', 'PolicyArn': 'arn:aws:iam::166151834296:policy/AmazonBedrockS3PolicyForKnowledgeBase_1232323'}, {'PolicyName': 'AmazonBedrockOSSPolicyForKnowledgeBase_1232323', 'PolicyArn': 'arn:aws:iam::166151834296:policy/AmazonBedrockOSSPolicyForKnowledgeBase_1232323'}, {'PolicyName': 'AmazonBedrockCloudWatchPolicyForKnowledgeBase_1232323', 'PolicyArn': 'arn:aws:iam::166151834296:policy/AmazonBedrockCloudWatchPolicyForKnowledgeBase_1232323'}]
Detached policy AmazonBedrockFoundationModelPolicyForKnowledgeBase_1232323 from role AmazonBedrockExecutionRoleForKnowledgeBase_1232323


...

Deleted policy AmazonBedrockS3PolicyForKnowledgeBase_1232323 from role AmazonBedrockExecutionRoleForKnowledgeBase_1232323
Detached policy AmazonBedrockOSSPolicyForKnowledgeBase_1232323 from role AmazonBedrockExecutionRoleForKnowledgeBase_1232323


Deleted policy AmazonBedrockOSSPolicyForKnowledgeBase_1232323 from role AmazonBedrockExecutionRoleForKnowledgeBase_1232323
Detached policy AmazonBedrockCloudWatchPolicyForKnowledgeBase_1232323 from role AmazonBedrockExecutionRoleForKnowledgeBase_1232323


Deleted policy AmazonBedrockCloudWatchPolicyForKnowledgeBase_1232323 from role AmazonBedrockExecutionRoleForKnowledgeBase_1232323


Deleted role AmazonBedrockExecutionRoleForKnowledgeBase_1232323
======== All IAM roles and policies deleted =========


======== Vector Index, collection and associated policies deleted =========
